# Twi ↔ English Translation Evaluation Suite

Comprehensive evaluation notebook for assessing Twi-English translation models.

**Metrics:**
- **BLEU** — Standard MT benchmark (higher is better)
- **chrF** — Character F-score, best for morphologically rich languages like Twi (higher is better)
- **TER** — Translation Edit Rate (lower is better)

**Evaluation Modes:**
- Full test set evaluation
- Direction-separated evaluation (Twi→En vs En→Twi)
- Error analysis with best/worst examples
- Comparison against v1 baseline

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece \
             accelerate matplotlib seaborn pandas

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

# Model to evaluate (change these to your trained models)
TWI_TO_EN_MODEL = "ninte/twi-en-nllb-v2"    # or local path: "./twi-en-nllb-v2"
EN_TO_TWI_MODEL = "ninte/en-twi-nllb-v2"    # or local path: "./en-twi-nllb-v2"

# Alternatively, use the v1 model for comparison
V1_MODEL = "ninte/twi-en-marianmt"

# Evaluation settings
MAX_LEN = 256
BATCH_SIZE = 32  # For batched evaluation

# v1 Baseline scores (for comparison)
V1_BASELINE = {
    'twi_to_en': {'bleu': 10.50, 'chrf': 32.30, 'ter': 87.64},
    'en_to_twi': {'bleu': 6.32, 'chrf': 28.74, 'ter': 91.55},
    'overall':   {'bleu': 8.26, 'chrf': 30.36, 'ter': 89.90},
}

print("Configuration loaded.")

## 3. Load Test Data

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load datasets
print("Loading datasets...")
ds1 = load_dataset("Ghana-NLP/TWI_ENGLISH_PARALLEL_TEXT")
ds2 = load_dataset("Ghana-NLP/ENGLISH_TWI_PARALLEL_TEXT")

df1 = ds1['train'].to_pandas().rename(columns={'text': 'twi', 'label': 'english'})
df2 = ds2['train'].to_pandas().rename(columns={'text': 'english', 'label': 'twi'})
df = pd.concat([df1[['twi', 'english']], df2[['twi', 'english']]], ignore_index=True)

# Clean
TWI_SPECIAL = set('ɛɔŋƐƆŊ')

def clean_text(text):
    if not isinstance(text, str) or pd.isna(text):
        return None
    text = re.sub(r'^\d+\.\s*', '', text.strip())
    text = re.sub(r' +', ' ', text)
    return text.strip() or None

def is_twi_text(text):
    return any(c in TWI_SPECIAL for c in str(text))

df['twi'] = df['twi'].apply(clean_text)
df['english'] = df['english'].apply(clean_text)
df = df.dropna(subset=['twi', 'english'])

# Length filter
df['twi_wc'] = df['twi'].str.split().str.len()
df['eng_wc'] = df['english'].str.split().str.len()
df = df[(df['twi_wc'] >= 3) & (df['eng_wc'] >= 3)]
df = df[(df['twi_wc'] <= 150) & (df['eng_wc'] <= 150)]

# Remove swapped pairs
df = df[df['twi'].apply(lambda x: not all(ord(c) < 128 for c in str(x).replace(' ', '')))]
df = df.drop_duplicates(subset=['twi', 'english'])
df = df.drop(columns=['twi_wc', 'eng_wc']).reset_index(drop=True)

# Create test split (same random_state for reproducibility)
_, temp_df = train_test_split(df, test_size=0.2, random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Direction-separate test set
test_df['is_twi_source'] = test_df['twi'].apply(is_twi_text)
twi_to_en_test = test_df[test_df['is_twi_source']].copy()
en_to_twi_test = test_df[~test_df['is_twi_source']].copy()

print(f"\nTest set: {len(test_df):,} pairs")
print(f"  Twi→En: {len(twi_to_en_test):,}")
print(f"  En→Twi: {len(en_to_twi_test):,}")

## 4. Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def load_model(model_id):
    """Load model and tokenizer."""
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    print(f"  Device: {device}")
    print(f"  Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
    
    return model, tokenizer, device

# Load the Twi→En model (can change to evaluate different models)
model, tokenizer, device = load_model(TWI_TO_EN_MODEL)

## 5. Evaluation Functions

In [ ]:
import sacrebleu as sb
from tqdm import tqdm

def evaluate_model(model, tokenizer, df, 
                   src_col='twi', ref_col='english',
                   src_lang='twi_Latn', tgt_lang='eng_Latn',
                   batch_size=32, label='Evaluation'):
    """
    Full evaluation with BLEU, chrF, TER.
    Works for both NLLB and MarianMT models.
    """
    model.eval()
    preds, refs, sources = [], [], []
    
    # Check if NLLB model (has lang tokens)
    is_nllb = hasattr(tokenizer, 'src_lang')
    tgt_token_id = None
    if is_nllb:
        try:
            tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
            if tgt_token_id == tokenizer.unk_token_id:
                tgt_token_id = None
        except:
            pass
    
    # Batch evaluation
    for i in tqdm(range(0, len(df), batch_size), desc=label):
        batch = df.iloc[i:i+batch_size]
        
        if is_nllb:
            tokenizer.src_lang = src_lang
        
        inputs = tokenizer(
            list(batch[src_col]),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        gen_kwargs = {
            'num_beams': 4,
            'max_length': MAX_LEN,
            'no_repeat_ngram_size': 3,
            'length_penalty': 1.0,
            'early_stopping': True,
        }
        if tgt_token_id is not None:
            gen_kwargs['forced_bos_token_id'] = tgt_token_id
        
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(decoded)
        refs.extend(list(batch[ref_col]))
        sources.extend(list(batch[src_col]))
    
    # Compute metrics
    bleu = sb.corpus_bleu(preds, [refs])
    chrf = sb.corpus_chrf(preds, [refs])
    ter = sb.corpus_ter(preds, [refs])
    
    # Print results
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  Pairs: {len(df)}")
    print(f"{'='*60}")
    print(f"  BLEU        : {bleu.score:.2f}")
    print(f"  chrF        : {chrf.score:.2f}")
    print(f"  TER         : {ter.score:.2f}  (lower = better)")
    print(f"  BLEU 1-gram : {bleu.precisions[0]:.2f}")
    print(f"  BLEU 2-gram : {bleu.precisions[1]:.2f}")
    print(f"  BLEU 3-gram : {bleu.precisions[2]:.2f}")
    print(f"  BLEU 4-gram : {bleu.precisions[3]:.2f}")
    print(f"  BP          : {bleu.bp:.4f}")
    print(f"{'='*60}\n")
    
    return {
        'bleu': round(bleu.score, 2),
        'chrf': round(chrf.score, 2),
        'ter': round(ter.score, 2),
        'bp': round(bleu.bp, 4),
        'preds': preds,
        'refs': refs,
        'sources': sources,
        'n': len(df),
    }

## 6. Run Evaluation

In [ ]:
# Full test set evaluation
results_all = evaluate_model(
    model, tokenizer, test_df,
    src_col='twi', ref_col='english',
    src_lang='twi_Latn', tgt_lang='eng_Latn',
    label='Full Test Set (Mixed)'
)

In [ ]:
# Direction-separated evaluation: Twi → English
if len(twi_to_en_test) > 0:
    results_twi_en = evaluate_model(
        model, tokenizer, twi_to_en_test,
        src_col='twi', ref_col='english',
        src_lang='twi_Latn', tgt_lang='eng_Latn',
        label='Twi → English'
    )
else:
    print("No Twi→En test pairs available.")

## 7. Comparison with v1 Baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_comparison(v1_scores, v2_scores, direction='overall'):
    """Plot comparison between v1 and v2 scores."""
    metrics = ['BLEU', 'chrF']
    v1_vals = [v1_scores['bleu'], v1_scores['chrf']]
    v2_vals = [v2_scores['bleu'], v2_scores['chrf']]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - width/2, v1_vals, width, label='v1 (MarianMT)', color='#ff9999')
    bars2 = ax.bar(x + width/2, v2_vals, width, label='v2 (NLLB)', color='#66b3ff')
    
    ax.set_ylabel('Score')
    ax.set_title(f'Model Comparison: {direction}')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    
    # Add value labels
    for bar, val in zip(bars1, v1_vals):
        ax.annotate(f'{val:.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10)
    for bar, val in zip(bars2, v2_vals):
        ax.annotate(f'{val:.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/comparison_{direction}.png', dpi=150)
    plt.show()

# Compare overall scores
print("\n" + "="*60)
print("  v1 vs v2 COMPARISON")
print("="*60)
print(f"\n  {'Metric':<10} {'v1 (MarianMT)':<15} {'v2 (NLLB)':<15} {'Δ':>8}")
print(f"  {'-'*48}")

for metric in ['bleu', 'chrf', 'ter']:
    v1_val = V1_BASELINE['overall'][metric]
    v2_val = results_all[metric]
    delta = v2_val - v1_val
    better = '↑' if (delta > 0 and metric != 'ter') or (delta < 0 and metric == 'ter') else '↓'
    print(f"  {metric.upper():<10} {v1_val:<15.2f} {v2_val:<15.2f} {delta:>+7.2f} {better}")

print("="*60)

# Plot comparison
plot_comparison(V1_BASELINE['overall'], results_all, 'Overall')

## 8. Error Analysis: Best and Worst Translations

In [ ]:
def show_examples(preds, refs, sources, n_each=5):
    """Print best and worst n_each translations by sentence BLEU."""
    scored = []
    for pred, ref, src in zip(preds, refs, sources):
        score = sb.sentence_bleu(pred, [ref]).score
        scored.append((score, src, pred, ref))
    scored.sort(key=lambda x: x[0])
    
    print(f"\n{'─'*70}")
    print(f"  TOP {n_each} TRANSLATIONS (Highest BLEU)")
    print(f"{'─'*70}")
    for score, src, pred, ref in reversed(scored[-n_each:]):
        print(f"\n  Source : {src[:100]}{'...' if len(src) > 100 else ''}")
        print(f"  Pred   : {pred[:100]}{'...' if len(pred) > 100 else ''}")
        print(f"  Ref    : {ref[:100]}{'...' if len(ref) > 100 else ''}")
        print(f"  BLEU   : {score:.1f}")
    
    print(f"\n{'─'*70}")
    print(f"  BOTTOM {n_each} TRANSLATIONS (Lowest BLEU)")
    print(f"{'─'*70}")
    for score, src, pred, ref in scored[:n_each]:
        print(f"\n  Source : {src[:100]}{'...' if len(src) > 100 else ''}")
        print(f"  Pred   : {pred[:100]}{'...' if len(pred) > 100 else ''}")
        print(f"  Ref    : {ref[:100]}{'...' if len(ref) > 100 else ''}")
        print(f"  BLEU   : {score:.1f}")

show_examples(
    results_all['preds'],
    results_all['refs'],
    results_all['sources'],
    n_each=5
)

## 9. BLEU Score Distribution

In [ ]:
# Calculate sentence-level BLEU scores
sentence_bleus = [
    sb.sentence_bleu(pred, [ref]).score
    for pred, ref in zip(results_all['preds'], results_all['refs'])
]

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sentence_bleus, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=np.mean(sentence_bleus), color='red', linestyle='--', 
                label=f'Mean: {np.mean(sentence_bleus):.1f}')
axes[0].axvline(x=np.median(sentence_bleus), color='orange', linestyle='--',
                label=f'Median: {np.median(sentence_bleus):.1f}')
axes[0].set_xlabel('Sentence BLEU Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sentence-Level BLEU Scores')
axes[0].legend()

# Score buckets
buckets = {
    '0-10 (Poor)': sum(1 for s in sentence_bleus if s < 10),
    '10-20 (Fair)': sum(1 for s in sentence_bleus if 10 <= s < 20),
    '20-30 (Good)': sum(1 for s in sentence_bleus if 20 <= s < 30),
    '30-50 (Very Good)': sum(1 for s in sentence_bleus if 30 <= s < 50),
    '50+ (Excellent)': sum(1 for s in sentence_bleus if s >= 50),
}
colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff', '#9b59b6']
axes[1].pie(buckets.values(), labels=buckets.keys(), autopct='%1.1f%%', colors=colors)
axes[1].set_title('BLEU Score Distribution by Quality Tier')

plt.tight_layout()
plt.savefig('/kaggle/working/bleu_distribution.png', dpi=150)
plt.show()

print(f"\nBLEU Score Statistics:")
print(f"  Mean   : {np.mean(sentence_bleus):.2f}")
print(f"  Median : {np.median(sentence_bleus):.2f}")
print(f"  Std    : {np.std(sentence_bleus):.2f}")
print(f"  Min    : {np.min(sentence_bleus):.2f}")
print(f"  Max    : {np.max(sentence_bleus):.2f}")

## 10. Length Analysis

In [ ]:
# Analyze how BLEU correlates with sentence length
source_lengths = [len(s.split()) for s in results_all['sources']]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(source_lengths, sentence_bleus, alpha=0.3, c=sentence_bleus, cmap='viridis')
ax.set_xlabel('Source Sentence Length (words)')
ax.set_ylabel('Sentence BLEU Score')
ax.set_title('BLEU Score vs. Sentence Length')
plt.colorbar(scatter, label='BLEU Score')

# Add trend line
z = np.polyfit(source_lengths, sentence_bleus, 1)
p = np.poly1d(z)
x_line = np.linspace(min(source_lengths), max(source_lengths), 100)
ax.plot(x_line, p(x_line), 'r--', label=f'Trend (slope: {z[0]:.2f})')
ax.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/length_analysis.png', dpi=150)
plt.show()

# Correlation
corr = np.corrcoef(source_lengths, sentence_bleus)[0, 1]
print(f"Correlation between length and BLEU: {corr:.3f}")

## 11. Summary Report

In [ ]:
print("\n" + "═"*70)
print("  EVALUATION SUMMARY REPORT")
print("═"*70)
print(f"\n  Model Evaluated: {TWI_TO_EN_MODEL}")
print(f"  Test Set Size  : {len(test_df):,} pairs")
print(f"\n  ── Overall Metrics ──")
print(f"    BLEU : {results_all['bleu']:.2f}")
print(f"    chrF : {results_all['chrf']:.2f}")
print(f"    TER  : {results_all['ter']:.2f}")

if 'results_twi_en' in dir() and results_twi_en:
    print(f"\n  ── Twi → English ──")
    print(f"    BLEU : {results_twi_en['bleu']:.2f}")
    print(f"    chrF : {results_twi_en['chrf']:.2f}")
    print(f"    TER  : {results_twi_en['ter']:.2f}")

print(f"\n  ── v1 Baseline Comparison ──")
bleu_imp = results_all['bleu'] - V1_BASELINE['overall']['bleu']
chrf_imp = results_all['chrf'] - V1_BASELINE['overall']['chrf']
ter_imp = V1_BASELINE['overall']['ter'] - results_all['ter']  # Lower is better

print(f"    BLEU improvement: {bleu_imp:+.2f} ({100*bleu_imp/V1_BASELINE['overall']['bleu']:+.1f}%)")
print(f"    chrF improvement: {chrf_imp:+.2f} ({100*chrf_imp/V1_BASELINE['overall']['chrf']:+.1f}%)")
print(f"    TER  improvement: {ter_imp:+.2f} ({100*ter_imp/V1_BASELINE['overall']['ter']:+.1f}%)")

print(f"\n  ── Score Interpretation ──")
if results_all['bleu'] < 10:
    quality = "Below baseline — needs investigation"
elif results_all['bleu'] < 20:
    quality = "At or above v1 baseline — acceptable"
elif results_all['bleu'] < 30:
    quality = "Good — usable in downstream applications"
else:
    quality = "Excellent for this low-resource pair!"
print(f"    {quality}")

print("\n" + "═"*70)

## 12. Save Results

In [ ]:
import json

# Save evaluation results
results_summary = {
    'model': TWI_TO_EN_MODEL,
    'test_size': len(test_df),
    'overall': {
        'bleu': results_all['bleu'],
        'chrf': results_all['chrf'],
        'ter': results_all['ter'],
    },
    'v1_baseline': V1_BASELINE['overall'],
    'improvement': {
        'bleu': round(results_all['bleu'] - V1_BASELINE['overall']['bleu'], 2),
        'chrf': round(results_all['chrf'] - V1_BASELINE['overall']['chrf'], 2),
        'ter': round(V1_BASELINE['overall']['ter'] - results_all['ter'], 2),
    }
}

with open('/kaggle/working/evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

# Save predictions for analysis
predictions_df = pd.DataFrame({
    'source': results_all['sources'],
    'prediction': results_all['preds'],
    'reference': results_all['refs'],
    'sentence_bleu': sentence_bleus,
})
predictions_df.to_csv('/kaggle/working/predictions.csv', index=False)

print("Results saved:")
print("  - /kaggle/working/evaluation_results.json")
print("  - /kaggle/working/predictions.csv")
print("  - /kaggle/working/comparison_Overall.png")
print("  - /kaggle/working/bleu_distribution.png")
print("  - /kaggle/working/length_analysis.png")